# 623 SPP: independent direct target+fill LSTM vs one-filter CNN

Both models receive only cache-line addresses, exactly the effective input read by `spp_dev2.cc`. The 128-class head owns 64 offsets × L2/LLC fill. Training utility comes from future demand reuse; captured SPP actions are comparator/budget data only. The CNN is exactly one causal kernel-3, stride-1 filter.

In [ ]:
import hashlib, os, pathlib, shutil, subprocess, sys, tarfile, torch
from google.colab import userdata
assert torch.cuda.is_available(), 'Select a GPU runtime (A100 preferred)'
torch.set_float32_matmul_precision('high')
torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False
REPO='/content/cache_arch'; TOKEN=userdata.get('GITHUB_TOKEN')
assert TOKEN, 'Add GITHUB_TOKEN to Colab Secrets'
ASKPASS='/content/cache_arch_git_askpass.sh'
pathlib.Path(ASKPASS).write_text('#!/bin/sh\ncase "$1" in *Username*) echo x-access-token ;; *) echo "$GITHUB_TOKEN" ;; esac\n'); os.chmod(ASKPASS,0o700)
env=os.environ.copy(); env.update({'GIT_ASKPASS':ASKPASS,'GIT_TERMINAL_PROMPT':'0','GITHUB_TOKEN':TOKEN})
try:
    if not os.path.isdir(REPO): subprocess.run(['git','clone','https://github.com/Angelawoo572/cache_arch.git',REPO],check=True,env=env)
    else: subprocess.run(['git','-C',REPO,'pull','--ff-only','origin','main'],check=True,env=env)
finally: pathlib.Path(ASKPASS).unlink(missing_ok=True)
print(torch.cuda.get_device_name(0),subprocess.check_output(['git','-C',REPO,'rev-parse','HEAD'],text=True).strip())

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')
RUN_ID='623_offline_lstm_cnn_spp_direct_v4_seed7'; DRIVE_ROOT=f'/content/drive/MyDrive/cache_prefetch_623_spp_direct/{RUN_ID}'
INPUT_DIR=f'{DRIVE_ROOT}/colab_input'; OUTPUT_ROOT=f'{DRIVE_ROOT}/colab_output'; os.makedirs(DRIVE_ROOT,exist_ok=True)
name=f'{RUN_ID}.colab_input.tar.gz'; uploaded=files.upload(); assert name in uploaded,f'Select {name}'
archive=f'{DRIVE_ROOT}/{name}'; pathlib.Path(archive).write_bytes(uploaded[name])
if os.path.isdir(INPUT_DIR): shutil.rmtree(INPUT_DIR)
os.makedirs(INPUT_DIR,exist_ok=True)
with tarfile.open(archive,'r:gz') as handle: handle.extractall(INPUT_DIR)
for record in pathlib.Path(f'{INPUT_DIR}/SHA256SUMS').read_text().splitlines():
    expected,item=record.split(maxsplit=1); item=item.lstrip('*')
    assert hashlib.sha256(pathlib.Path(f'{INPUT_DIR}/{item}').read_bytes()).hexdigest()==expected
print('verified',archive)

In [ ]:
import json
TRACE='623.xalancbmk_s-700B'; POLICY='spp'; ROLES=('train','guard','eval')
INPUTS={role:{'stream':f'{INPUT_DIR}/{TRACE}.{POLICY}.{role}_stream.csv.gz','teacher':f'{INPUT_DIR}/{TRACE}.{POLICY}.{role}_teacher_actions.csv.gz'} for role in ROLES}
for items in INPUTS.values():
 for path in items.values(): assert os.path.isfile(path),path
manifest=json.loads(pathlib.Path(f'{INPUT_DIR}/collection_manifest.json').read_text())
expected={'status':'PASS','experiment_revision':'spp_direct_io_sliding_cnn_v4_independent_utility','neural_role':'standalone_direct_action_prefetcher','source_decision_effective_external_input':['addr'],'same_external_input_contract':True,'normal_policy_outputs_used_as_model_inputs':False,'normal_policy_candidates_used_as_model_inputs':False,'normal_policy_private_state_used_as_model_inputs':False}
bad={k:(manifest.get(k),v) for k,v in expected.items() if manifest.get(k)!=v}; assert not bad,bad
SCRIPT=f'{REPO}/formal_NN_training/experiments/623_offline_lstm_cnn_spp/python/train_and_offline_infer.py'
SOURCE=f'{INPUT_DIR}/spp_source_contract.json'

In [ ]:
LOCAL_OUTPUT=f'/content/{RUN_ID}_colab_output'
if os.path.isdir(LOCAL_OUTPUT): shutil.rmtree(LOCAL_OUTPUT)
os.makedirs(LOCAL_OUTPUT)
SPECS=[
 {'tag':'direct_spp_lstm_h4','family':'lstm','size':4,'pair':'p0','parameters':880},
 {'tag':'direct_spp_cnn_c5','family':'cnn','size':5,'pair':'p0','parameters':908},
 {'tag':'direct_spp_lstm_h8','family':'lstm','size':8,'pair':'p1','parameters':1760},
 {'tag':'direct_spp_cnn_c10','family':'cnn','size':10,'pair':'p1','parameters':1688},
 {'tag':'direct_spp_lstm_h16','family':'lstm','size':16,'pair':'p2','parameters':3904},
 {'tag':'direct_spp_cnn_c24','family':'cnn','size':24,'pair':'p2','parameters':3872},
]
SWEEP=[]
for spec in SPECS:
 out=f"{LOCAL_OUTPUT}/{spec['tag']}"; cmd=[sys.executable,SCRIPT,'--policy',POLICY]
 for role in ROLES: cmd += [f'--{role}-stream',INPUTS[role]['stream'],f'--{role}-teacher-actions',INPUTS[role]['teacher']]
 cmd += ['--source-contract',SOURCE,'--out-dir',out,'--model-family',spec['family'],'--model-size',str(spec['size']),'--pair-id',spec['pair'],'--device','cuda','--seed','7','--epochs','8','--chunk-len','1024','--accumulate-chunks','16','--min-lead','4','--max-lead','256','--l2-lead-cutoff','64']
 print('\nTraining',spec['tag'],' '.join(cmd),flush=True); subprocess.run(cmd,check=True)
 meta=json.loads(pathlib.Path(f'{out}/run_metadata.json').read_text())
 expected={'model_tag':spec['tag'],'parameter_count':spec['parameters'],'matched_normal_prefetcher':POLICY,'neural_role':'standalone_direct_action_prefetcher','same_external_input_contract':True,'normal_policy_outputs_used_as_model_inputs':False,'normal_policy_candidates_used_as_model_inputs':False,'normal_policy_private_state_used_as_model_inputs':False,'experiment_revision':'spp_direct_io_sliding_cnn_v4_independent_utility'}
 bad={k:(meta.get(k),v) for k,v in expected.items() if meta.get(k)!=v}; assert not bad,bad
 SWEEP.append({k:meta[k] for k in ('model_tag','model_family','model_size','architecture_pair_id','parameter_count','threshold','offline_normal_entries','offline_nn_entries','eval_future_use_utility')})
if os.path.isdir(OUTPUT_ROOT): shutil.rmtree(OUTPUT_ROOT)
shutil.copytree(LOCAL_OUTPUT,OUTPUT_ROOT)
pathlib.Path(f'{OUTPUT_ROOT}/sweep_manifest.json').write_text(json.dumps({'trace':TRACE,'revision':'spp_direct_io_sliding_cnn_v4_independent_utility','points':SWEEP},indent=2)+'\n')
print(json.dumps(SWEEP,indent=2))

In [ ]:
OUTPUT_ARCHIVE=f'{DRIVE_ROOT}/{RUN_ID}.colab_output.tar.gz'
with tarfile.open(OUTPUT_ARCHIVE,'w:gz') as archive:
 for item in pathlib.Path(OUTPUT_ROOT).iterdir(): archive.add(item,arcname=item.name)
print('DONE',OUTPUT_ARCHIVE,os.path.getsize(OUTPUT_ARCHIVE),'bytes')

Copy the output archive to the matching server run and launch replay. Teacher actions are never neural inputs or labels.